# Experiment 2: Support Vector Machines
**Machine Learning | Autumn 2026**  
**Release:** October 13, 2026 | **Deadline:** November 11, 2026

Implement linear and nonlinear SVMs, inspect their hyperparameters, evaluate a
real dataset correctly, and implement a linear SVM using NumPy.

Tasks 1-4 are **required**. Task 5 is **optional**.

Use Python and write the report in English. Submit your completed `.ipynb`
(with outputs) and `report.pdf`; use `Template.doc` for the report structure.
Work independently and explain the results, rather than submitting plots alone.

- Tasks 1-3 and 5: scikit-learn is allowed.
- Task 4: implement training, loss, gradients, and prediction with NumPy only.
  Pandas/Matplotlib may be used for loading/plotting, and sklearn only for the reference model.
- Fill the marked blocks and the report questions. `NotImplementedError` identifies
  unfinished code deliberately; it is not an installation problem.
- Start Jupyter from the folder containing this notebook and the two CSV files.
  Select a Python kernel with NumPy, pandas, Matplotlib, SciPy, scikit-learn,
  seaborn and ipykernel installed; restart it before a final Run All.
  No GPU or dataset download is required.

If packages are missing, install them in the Python environment used by your
notebook: `python -m pip install numpy pandas matplotlib scipy scikit-learn
seaborn ipykernel jupyterlab`. Start Jupyter with `python -m jupyterlab`.


In [ ]:
import time
import platform
import sys
import importlib.metadata as metadata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, make_scorer, classification_report,
                             confusion_matrix)

SEED = 42
N_JOBS = 2
GRID_RESOLUTION = 160
DATA_DIR = Path.cwd()
assert all((DATA_DIR / f).is_file() for f in ("T1_data.csv", "T2_data.csv")), (
    "Start Jupyter in the folder containing this notebook and both CSV files.")
task_times = {}
notebook_started = time.perf_counter()
print("Python:", sys.version.split()[0], "| CPU:", platform.processor())
for package in ["numpy", "pandas", "matplotlib", "scipy", "scikit-learn", "seaborn"]:
    print(f"{package}: {metadata.version(package)}")


### Provided plotting helper
The same helper works for sklearn models, Pipelines, and the NumPy model. It receives its data and axes explicitly; it never uses task-global features or labels.


In [ ]:
def plot_decision_boundary(model, X, y, ax, title="", support_vectors=None):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y)
    lower, upper = X.min(axis=0), X.max(axis=0)
    padding = np.maximum((upper - lower) * 0.08, 0.1)
    xx, yy = np.meshgrid(
        np.linspace(lower[0] - padding[0], upper[0] + padding[0], GRID_RESOLUTION),
        np.linspace(lower[1] - padding[1], upper[1] + padding[1], GRID_RESOLUTION))
    mesh = np.column_stack([xx.ravel(), yy.ravel()])
    predictions = model.predict(mesh).reshape(xx.shape)
    ax.contourf(xx, yy, predictions, levels=[-2, 0, 2],
                colors=["#f9d5d5", "#c9e2f5"], alpha=0.65)
    scores = model.decision_function(mesh).reshape(xx.shape)
    ax.contour(xx, yy, scores, levels=[-1, 0, 1], colors="black",
               linestyles=["--", "-", "--"], linewidths=0.8)
    for label, color in [(-1, "#ba3434"), (1, "#236a9a")]:
        mask = y == label
        ax.scatter(X[mask, 0], X[mask, 1], c=color, s=22,
                   edgecolors="black", linewidths=0.3, label=str(label))
    if support_vectors is not None:
        support_vectors = np.asarray(support_vectors)
        ax.scatter(support_vectors[:, 0], support_vectors[:, 1],
                   facecolors="none", edgecolors="black", s=80,
                   linewidths=0.8, label="Support vectors")
    ax.set(xlabel="Feature 1", ylabel="Feature 2", title=title)
    ax.legend(fontsize=8)


## Task 1 Required: Linear SVM
For labels $y_i\in\{-1,+1\}$, the soft-margin primal is
$$
\min_{w,b,\xi}\frac12\|w\|^2+C\sum_i\xi_i,
\quad y_i(w^T x_i+b)\geq1-\xi_i,\quad\xi_i\geq0.
$$
Load `T1_data.csv`: 51 rows, two features, and a label column. Visualize the
data, fit a linear SVM, and show the decision boundary and the two margin contours.
Compare $C=0.01,0.1,1,10,100,1000$. These are full-dataset illustrations;
training accuracy is not an estimate of performance on unseen data.


In [ ]:
task_started = time.perf_counter()
# Load the CSV into X_t1 (ndarray) and y_t1 (one-dimensional ndarray).
########## Start of Your Code ##########
raise NotImplementedError("Task 1: load T1_data.csv")
########## End of Your Code ############
assert X_t1.shape == (51, 2) and y_t1.shape == (51,)
assert set(np.unique(y_t1)) == {-1, 1}
print(X_t1.shape, y_t1.shape)


In [ ]:
# Scatter plot; fit clf_linear = SVC with a linear kernel and C=1.
# Use the provided helper to show its boundary, margins, and support vectors.
########## Start of Your Code ##########
raise NotImplementedError("Task 1: plot and fit the linear SVM")
########## End of Your Code ############


In [ ]:
C_t1 = [0.01, 0.1, 1, 10, 100, 1000]
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
########## Start of Your Code ##########
raise NotImplementedError("Task 1: compare the six C values")
########## End of Your Code ############
fig.tight_layout()
plt.show()
task_times["Task 1"] = time.perf_counter() - task_started


**Report:** Explain the effect of C using your plots. Identify support vectors and explain why a better fit to the training points is not sufficient evidence of better generalization.


## Task 2 Required: Nonlinear SVM
The soft-margin dual can be written as
$$
\min_\alpha\frac12\sum_{i,j}\alpha_i\alpha_jy_iy_jK(x_i,x_j)-\sum_i\alpha_i,
\quad\sum_i\alpha_i y_i=0,\quad0\leq\alpha_i\leq C.
$$
For the RBF kernel,
$$
K(x,z)=\exp(-\gamma\|x-z\|^2),\qquad\gamma=\frac{1}{2\sigma^2}.
$$
Gamma and sigma are different parameterizations: larger gamma means a narrower
kernel. Load the 200 rows of `T2_data.csv`. Fit an RBF SVM and visualize the
nine combinations of $C=[0.1,1,10]$ and $\gamma=[0.1,1,10]$.


In [ ]:
task_started = time.perf_counter()
# Load X_t2, y_t2; fit clf_rbf with C=1 and gamma=1; draw its boundary.
########## Start of Your Code ##########
raise NotImplementedError("Task 2: load, fit, and visualize")
########## End of Your Code ############
assert X_t2.shape == (200, 2) and y_t2.shape == (200,)
assert set(np.unique(y_t2)) == {-1, 1}


In [ ]:
C_visual = [0.1, 1, 10]
gamma_visual = [0.1, 1, 10]
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
########## Start of Your Code ##########
raise NotImplementedError("Task 2: plot the nine C/gamma combinations")
########## End of Your Code ############
fig.tight_layout()
plt.show()


### Cross-validation grid search
Use all 21 values $2^{-10},2^{-9},\ldots,2^{10}$ for **each** of C and gamma
(441 combinations, 2205 fold fits). Use five-fold stratified cross-validation
with shuffle enabled and random state 42; score accuracy. Report `best_params_`
and `best_score_` and plot the mean validation accuracy as a 21-by-21 heatmap.
This full-data CV is a tuning demonstration. Its best score is **not** an
independent test score; Task 5 demonstrates a separate held-out test protocol.


In [ ]:
# Use the specified 21-by-21 grid and stratified five-fold CV.
# Store the fitted search in grid_search; use n_jobs=N_JOBS.
########## Start of Your Code ##########
raise NotImplementedError("Task 2: grid search")
########## End of Your Code ############
print("Best parameters:", grid_search.best_params_)
print(f"Best tuning CV accuracy: {grid_search.best_score_:.4f}")


In [ ]:
# Plot mean CV accuracy as a 21-by-21 heatmap with labelled C/gamma axes.
########## Start of Your Code ##########
raise NotImplementedError("Task 2: score heatmap")
########## End of Your Code ############
task_times["Task 2"] = time.perf_counter() - task_started


**Report:** Explain the different roles of C and gamma; interpret your boundary grid and score heatmap. Report the best CV score and explain its limitation. Trying a polynomial or sigmoid kernel is optional.


## Task 3 Required: SVM on the Wisconsin Breast Cancer Dataset
Load `load_breast_cancer()` (569 samples, 30 features). Retain its original
labels: **0 = malignant**, **1 = benign**. Here malignant is the positive class
for precision, recall and F1, so use `pos_label=0` explicitly.

Split 80%/20% with `stratify=y` and `random_state=42`. Keep the raw features
in `X_train` and `X_test`. All model selection happens within the training set.
Standardization learns means/scales; place it **inside** the cross-validation
Pipeline so each validation fold remains unseen during fitting.


In [ ]:
task_started = time.perf_counter()
# Load cancer, X (DataFrame), y (Series); make the stratified 80/20 split.
########## Start of Your Code ##########
raise NotImplementedError("Task 3: load and split the data")
########## End of Your Code ############
assert isinstance(X, pd.DataFrame) and isinstance(y, pd.Series)
assert X.shape == (569, 30)
assert X_train.shape == (455, 30) and X_test.shape == (114, 30)
print("Original target names:", dict(enumerate(cancer.target_names)))
print("Training classes:", y_train.value_counts().sort_index().to_dict())
print("Test classes:", y_test.value_counts().sort_index().to_dict())


### Scaling demonstration
Create `X_train_scaled_demo` using only training data to check the transformation. This demonstration does not replace the raw `X_train` passed to GridSearchCV; the formal search must fit its own scaler inside each fold. Scaling equalizes feature units, but does not guarantee equal predictive importance.


In [ ]:
# Fit scaler_demo only on X_train; save its output separately.
########## Start of Your Code ##########
raise NotImplementedError("Task 3: demonstrate scaling without replacing raw X_train")
########## End of Your Code ############
assert np.allclose(X_train_scaled_demo.mean(axis=0), 0, atol=1e-7)
assert np.allclose(X_train_scaled_demo.std(axis=0), 1, atol=1e-7)


### Tune on the raw training features through a Pipeline
Use exactly seven values $10^{-3},\ldots,10^3$ for C and seven values
$10^{-4},\ldots,10^2$ for gamma. Use five-fold stratified CV with shuffle and
seed 42. Score **malignant F1**, using `make_scorer(f1_score, pos_label=0)`.
The Pipeline step names are `scaler` and `svc`, so parameter names are
`svc__C` and `svc__gamma`. Store the fitted search in `grid_search_cancer`.


In [ ]:
# Search a StandardScaler -> SVC Pipeline on RAW X_train.
# Fit using only y_train and score malignant (0) F1.
########## Start of Your Code ##########
raise NotImplementedError("Task 3: leakage-free Pipeline grid search")
########## End of Your Code ############
print("Best parameters:", grid_search_cancer.best_params_)
print(f"Best malignant F1 (tuning CV): {grid_search_cancer.best_score_:.4f}")


### Evaluate the selected model once on the test set
Report accuracy and malignant precision, recall and F1. With original label order `[0, 1]`, malignant false negatives appear in matrix row 0, column 1: true malignant predicted benign. This dataset experiment does not establish clinical suitability.


In [ ]:
# Predict on X_test once after model selection; explicitly score pos_label=0.
# Display all four metrics, both classes, and a labelled confusion matrix.
########## Start of Your Code ##########
raise NotImplementedError("Task 3: final test evaluation")
########## End of Your Code ############
task_times["Task 3"] = time.perf_counter() - task_started


**Report:** Explain why scaling matters, how the Pipeline prevents leakage, which class each metric describes, and what the false-negative count means. Discuss the limitations of this small test set.


## Task 4 Required: Implement a Linear SVM Using NumPy
Implement the hinge-loss objective with a **sum**, not a mean:
$$
L(w,b)=\frac12\|w\|^2+C\sum_i\max(0,1-y_i(w^Tx_i+b)).
$$
The bias is not regularized. Derive subgradients using the active set
$A=\{i:y_i(w^Tx_i+b)<1\}$; at exactly margin 1 choose the inactive branch.
Use the provided full-batch loop: initialize zero parameters, update 5000 times
with $\eta_t=0.05/\sqrt{t+1}$, and retain the lowest-loss parameters, including
the initial state. Raw losses may fluctuate; monotonic decrease is not required.

Complete `loss`, `subgradient`, the update, `decision_function`, and `predict`.
Prediction at score zero must be +1. Only NumPy may implement this model; do not
call sklearn estimators, automatic differentiation, or optimization libraries
inside the class. sklearn is allowed only for the reference comparison below.


In [ ]:
task_started = time.perf_counter()
# Provided: Task 4 reloads T1, without depending on overwritten task variables.
t4_data = pd.read_csv(DATA_DIR / "T1_data.csv")
X_t4_raw = t4_data.iloc[:, :2].to_numpy(dtype=float)
y_t4 = t4_data["label"].to_numpy(dtype=int)
t4_mean = X_t4_raw.mean(axis=0)
t4_scale = X_t4_raw.std(axis=0)
t4_scale = np.where(t4_scale == 0, 1, t4_scale)
X_t4 = (X_t4_raw - t4_mean) / t4_scale


In [ ]:
class NumPyLinearSVM:
    def __init__(self, C=1.0, learning_rate=0.05, n_iter=5000):
        if C <= 0 or learning_rate <= 0 or n_iter <= 0:
            raise ValueError("C, learning_rate, and n_iter must be positive")
        self.C = C
        self.learning_rate = learning_rate
        self.n_iter = n_iter

    def loss(self, X, y, w, b):
        ########## Start of Your Code ##########
        raise NotImplementedError("Complete this NumPy SVM block")
        ########## End of Your Code ############

    def subgradient(self, X, y, w, b):
        ########## Start of Your Code ##########
        raise NotImplementedError("Complete this NumPy SVM block")
        ########## End of Your Code ############

    def fit(self, X, y):
        X, y = np.asarray(X, dtype=float), np.asarray(y, dtype=float)
        if X.ndim != 2 or y.shape != (len(X),) or not np.isin(y, [-1, 1]).all():
            raise ValueError("Expected a feature matrix and one-dimensional -1/+1 labels")
        w, b = np.zeros(X.shape[1]), 0.0
        best_w, best_b = w.copy(), b
        self.best_loss_ = self.loss(X, y, w, b)
        self.loss_history_ = [self.best_loss_]
        for t in range(self.n_iter):
            grad_w, grad_b = self.subgradient(X, y, w, b)
            eta = self.learning_rate / np.sqrt(t + 1)
            ########## Start of Your Code ##########
            raise NotImplementedError("Complete this NumPy SVM block")
            ########## End of Your Code ############
            current_loss = self.loss(X, y, w, b)
            if not np.isfinite(current_loss):
                raise FloatingPointError("Non-finite loss; check gradients and update")
            self.loss_history_.append(current_loss)
            if current_loss < self.best_loss_:
                self.best_loss_ = current_loss
                best_w, best_b = w.copy(), b
        self.w_, self.b_ = best_w, best_b
        self.loss_history_ = np.asarray(self.loss_history_)
        return self

    def decision_function(self, X):
        ########## Start of Your Code ##########
        raise NotImplementedError("Complete this NumPy SVM block")
        ########## End of Your Code ############

    def predict(self, X):
        ########## Start of Your Code ##########
        raise NotImplementedError("Complete this NumPy SVM block")
        ########## End of Your Code ############


### Provided numerical gradient check
This checks a point away from hinge-loss kinks using central differences. It also checks prediction at zero. Passing this check is necessary, but does not alone prove that training is correct.


In [ ]:
check_model = NumPyLinearSVM(C=0.7)
check_X = np.array([[0.2, -0.3], [0.7, 0.4], [-0.5, 0.1]])
check_y = np.array([1, -1, 1])
eps = 1e-6
gradient_errors = []
# All-active, mixed-active, and no-active samples exercise all branches.
for theta in [np.array([0.1, -0.2, 0.05]), np.array([3.0, -2.0, 0.5]),
              np.array([-3.0, -4.0, 1.5])]:
    assert np.min(np.abs(check_y * (check_X @ theta[:2] + theta[2]) - 1)) > 0.1
    analytic_w, analytic_b = check_model.subgradient(check_X, check_y, theta[:2], theta[2])
    analytic = np.r_[analytic_w, analytic_b]
    numeric = np.empty(3)
    for j in range(3):
        delta = np.zeros(3)
        delta[j] = eps
        plus, minus = theta + delta, theta - delta
        numeric[j] = (check_model.loss(check_X, check_y, plus[:2], plus[2]) -
                      check_model.loss(check_X, check_y, minus[:2], minus[2])) / (2 * eps)
    assert np.allclose(analytic, numeric, atol=1e-6, rtol=1e-5)
    gradient_errors.append(float(np.max(np.abs(analytic - numeric))))
check_model.w_, check_model.b_ = np.zeros(2), 0.0
assert check_model.predict(np.zeros((1, 2)))[0] == 1
print("Maximum subgradient error:", max(gradient_errors))
print("Zero-score prediction: +1; gradient check passed.")


### Compare C = 0.1, 1, 10 against sklearn
Train both implementations on the same NumPy-standardized T1 data and use the
same C. For each model report training accuracy, objective value and geometric
margin width $2/\|w\|$ **in standardized coordinates**. Plot the NumPy loss
histories and six boundaries. The comparison is about optimization, not test
generalization. A finite-step solver need not reproduce sklearn's coefficients.


In [ ]:
C_manual = [0.1, 1.0, 10.0]
# Fit NumPyLinearSVM and reference SVC on X_t4, y_t4 for each C.
# Produce the comparison table, loss histories, and six boundary plots.
########## Start of Your Code ##########
raise NotImplementedError("Task 4: compare the implementations")
########## End of Your Code ############
task_times["Task 4"] = time.perf_counter() - task_started


**Report:** Derive both subgradients, explain your update and best-iterate rule, interpret the table and loss curves, and discuss why similar predictions do not require identical coefficients. Explain the sum-versus-mean convention and the coordinate system of the reported margin.


## Task 5 Optional: Generalization and Overfitting
Reload T2 and make a **new stratified 80%/20% split**, with seed 42. Do not use
the model or best hyperparameters from Task 2: that search saw all of T2.
Use `StandardScaler -> SVC` with fixed C=1 and
gamma in `[0.01, 0.1, 1, 10, 100]`.

On the training subset only, run shuffled stratified five-fold CV with seed 42
and `return_train_score=True`. Plot the mean training and validation accuracies
with validation standard deviations. Fit each candidate on the full training
subset to count its support vectors. Select gamma by mean CV accuracy (ties use
the first candidate), freeze that choice, and evaluate **only that model once**
on the test subset. Draw boundaries for the smallest, selected and largest gamma.
Discuss underfitting and overfitting without assuming strictly monotonic results.

If you skip this task, leave `RUN_TASK5=False`; its cells then run without errors.


In [ ]:
RUN_TASK5 = False  # Set True only when you implement the optional task.


In [ ]:
if RUN_TASK5:
    task_started = time.perf_counter()
    # Reload and split T2 independently; run the specified Pipeline CV.
    # Produce a candidate table, generalization curve and three boundaries.
    # Choose gamma using training-subset CV; predict on the test subset once.
    ########## Start of Your Code ##########
    raise NotImplementedError("Task 5: independent generalization experiment")
    ########## End of Your Code ############
    task_times["Task 5"] = time.perf_counter() - task_started


**Optional report:** Interpret the training/validation gap, variability across folds, support-vector counts and boundaries. Explain why the test set must not select gamma and why a 40-sample test accuracy is uncertain.


## Runtime record
These elapsed times include computation and figure generation after setup. Package installation, notebook kernel startup, and report writing are excluded.


In [ ]:
print("Per-task CPU execution time (seconds):")
display(pd.Series(task_times, name="seconds").round(3).to_frame())
total_runtime = time.perf_counter() - notebook_started
print(f"Notebook elapsed time after imports: {total_runtime:.3f} seconds")
